# IndicatorsEnv — Colab Test Notebook

Test the multi-step RL environment and run inference with **Qwen2.5-1.5B** via the HF Inference API.

**What this notebook does:**
1. Upload & extract the hackathon code
2. Install dependencies
3. Start the IndicatorsEnv server (FastAPI on port 7860)
4. Smoke test — verify 5-step episodes, grader, and baseline (no LLM needed)
5. Full inference — run `inference.py` with Qwen2.5-1.5B against the live environment

**Requirements:**
- A Hugging Face token with Inference API access (`HF_TOKEN`)
- A zip of the hackathon code (instructions in Step 1)

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")

%cd /content/hackathon
print("Working directory:", os.getcwd())
print("Files:", os.listdir("."))

## Step 1: Upload & Extract Code

Create the zip on your Mac first, then upload it here:

```bash
# Run this in your terminal (Mac)
cd "/Users/aaryanmanawat/Aaryan/StockAnalyzer Pro/version3.0/StockAnalyzer-Pro"
zip -r hackathon.zip hackathon/env hackathon/inference.py hackathon/requirements.txt
```

Then run the cell below and select `hackathon.zip` when the file picker appears.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload() # Select hackathon.zip from your Mac

with zipfile.ZipFile('hackathon.zip', 'r') as z:
    z.extractall('/content/')

%cd /content/hackathon
print(f'Ready in: {os.getcwd()}')

## Step 2: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn yfinance pandas numpy requests openai
print("Done")

## Step 3: Start the IndicatorsEnv Server

Starts FastAPI on `http://localhost:7860`. The first `reset()` call takes ~10–30s as yfinance fetches real NSE OHLCV data.

In [ ]:
import subprocess, time, sys, requests

server = subprocess.Popen(
    [sys.executable, "indicators_env.py"],
    cwd="/content/hackathon/env",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(12)

try:
    resp = requests.get("http://localhost:7860/health", timeout=5)
    print("Server:", resp.json())
except Exception as e:
    print("Not ready:", e)
    out = server.stdout.read1(2048).decode("utf-8", errors="replace")
    print("Startup log:", out)

## Step 4: Smoke Test (No LLM Required)

Verifies three things:
- **Multi-step structure**: `done=False` on steps 1–4, `done=True` on step 5
- **Grader fix**: returns `score > 0` with the correct `"predicted"` key
- **Baseline loop**: 10 episodes × 5 steps = 50 total steps, score ≈ 0.33

In [ ]:
import requests

BASE = "http://localhost:7860"

# ── Reset ─────────────────────────────────────────────────────────────────
r = requests.post(f"{BASE}/reset", params={"term": "short"}, timeout=60)
assert r.status_code == 200, f"Reset failed: {r.text}"
data       = r.json()
session_id = data["info"]["session_id"]
print(f"Reset OK — symbol={data['observation']['symbol']}  date={data['observation']['date']}")
print(f"Session: {session_id}\n")

# ── 5 Steps ───────────────────────────────────────────────────────────────
print("Steps:")
for i in range(1, 6):
    s      = requests.post(
        f"{BASE}/step",
        params={"session_id": session_id},
        json={"direction": "Bullish", "conviction": 0.7},
        timeout=30,
    ).json()
    step   = s["info"]["step"]
    done   = s["done"]
    reward = s["reward"]
    ok     = (i < 5 and not done) or (i == 5 and done)
    print(f"  step={step}  done={done}  reward={reward}  {'✅' if ok else '❌ WRONG'}")

# ── Grader ────────────────────────────────────────────────────────────────
print()
g = requests.post(f"{BASE}/grader", json={
    "task_id": "short_term_direction",
    "episode_results": [
        {"predicted": "Bullish", "ground_truth": "Bullish", "conviction": 0.7},
        {"predicted": "Bearish", "ground_truth": "Bearish", "conviction": 0.8},
        {"predicted": "Bullish", "ground_truth": "Neutral",  "conviction": 0.5},
    ],
}).json()
score = g["score"]
print(f"Grader score: {score}  {'✅' if score > 0 else '❌ WRONG — grader bug'}")

# ── Baseline (takes ~60s with real yfinance) ──────────────────────────────
print("\nRunning baseline — this takes ~60s...")
b         = requests.get(f"{BASE}/baseline", timeout=300).json()
num_eps   = b["tasks"]["short_term_direction"]["num_episodes"]
mean      = b["overall_mean"]
ok        = num_eps == 50
print(f"short_term num_episodes={num_eps}  {'✅' if ok else f'❌ WRONG (expected 50)'}")
print(f"overall_mean={mean}  (expect ~0.33 for random agent)")

## Step 5: Full Inference with Qwen2.5-1.5B

Runs `inference.py` end-to-end against the live environment using the HF Inference API.
No GPU required — the 1.5B model runs on HF's servers.

**Paste your HF token in the cell below.** Free tier is enough for `--n_episodes 1` (45 LLM calls total across 3 tasks).

In [ ]:
import os

# ── Paste your HF token below ─────────────────────────────────────────────
# Get it from: https://huggingface.co/settings/tokens  (read access is enough)
HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # ← replace this
# ─────────────────────────────────────────────────────────────────────────

assert HF_TOKEN != "hf_YOUR_TOKEN_HERE", "Replace HF_TOKEN with your real token first!"

os.environ["API_BASE_URL"] = "https://api-inference.huggingface.co/v1/"
os.environ["MODEL_NAME"]   = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["HF_TOKEN"]     = HF_TOKEN

!python /content/hackathon/inference.py \
    --env_url http://localhost:7860 \
    --n_episodes 1

### Expected Output

```
[START] task=short_term_direction env=IndicatorsEnv model=Qwen/Qwen2.5-1.5B-Instruct
[STEP] step=1 action=Bullish reward=1.0000 done=False error=None
[STEP] step=2 action=Neutral  reward=0.0000 done=False error=None
[STEP] step=3 action=Bullish  reward=1.0000 done=False error=None
[STEP] step=4 action=Bearish  reward=0.0000 done=False error=None
[STEP] step=5 action=Bullish  reward=1.0000 done=True  error=None
[END]  success=True steps=5 score=0.XXXX rewards=[...]

[START] task=medium_term_direction ...
... (5 more steps) ...
[END]  success=True steps=5 ...

[START] task=long_term_conviction ...
... (5 more steps) ...
[END]  success=True steps=5 ...
```

**What to check:**
- `done=False` on steps 1–4, `done=True` on step 5 ← multi-step fix
- `score > 0` in every `[END]` line ← grader key fix
- 3 tasks × 5 steps = 15 total `[STEP]` lines per `n_episodes=1` run